## SafeRouteAI — Synthetic Safety Dataset Generation (Hyderabad)

**Purpose**: Bootstrap accident risk scores for Hyderabad road segments using NCRB/ADSI national data until GHMC ward-level data is ingested into PostGIS.

**Output**: `accident_idx` column only — feeds `SegmentFeatures.accident_idx` in `scorer.py`.
Other features (lighting, crosswalk, road_quality, incident_score, weather_risk) come from separate pipelines.

**Known limitations**:
- Telangana state average used as base — will underestimate Hyderabad urban risk
- Spatial distribution is uniform within Hyderabad bbox — no real cluster structure
- Replace with GHMC ward-level data from data.telangana.gov.in as soon as available


In [1]:
import pandas as pd
import numpy as np

# ── Hyderabad bounding box (GHMC limits) ──────────────────────────
HYD_BBOX = {
    "lat_min": 17.20, "lat_max": 17.60,
    "lon_min": 78.20, "lon_max": 78.70,
}
N_POINTS = 500      # synthetic points across Hyderabad
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Step 1 — Load and inspect raw data

In [2]:
crime_df    = pd.read_csv("NCRB_Table_1A.1.csv")
accident_df = pd.read_csv("ADSI_Table_1A.2.csv")

crime_df.columns    = crime_df.columns.str.strip()
accident_df.columns = accident_df.columns.str.strip()

print("Crime columns:")
print(crime_df.columns.tolist())
print("\nAccident columns:")
print(accident_df.columns.tolist())

Crime columns:
['Sl. No.', 'State/UT', '2020', '2021', '2022', 'Mid-Year Projected Population (in Lakhs) (2022)', 'Rate of Cognizable Crimes (IPC) (2022)', 'Chargesheeting Rate (2022)']

Accident columns:
['Sl. No.', 'State/UT/City', 'Road Accidents - Cases', 'Road Accidents - Injured', 'Road Accidents - Died', 'Railway Accidents - Cases', 'Railway Accidents - Injured', 'Railway Accidents - Died', 'Railway Crossing Accidents - Cases', 'Railway Crossing Accidents - Injured', 'Railway Crossing Accidents - Died', 'Total Traffic Accidents - Cases', 'Total Traffic Accidents - Injured', 'Total Traffic Accidents - Died']


## Step 2 — Extract relevant columns

In [3]:
crime_df = crime_df[[
    "State/UT",
    "Rate of Cognizable Crimes (IPC) (2022)"
]]

# Including Deaths for severity weighting — not just case counts
accident_df = accident_df[[
    "State/UT/City",
    "Total Traffic Accidents - Cases",
    "Total Traffic Accidents - Died",
]]
accident_df.rename(columns={"State/UT/City": "State/UT"}, inplace=True)

df = pd.merge(crime_df, accident_df, on="State/UT")
print(f"Merged shape: {df.shape}")
print(f"\nTelangana row:")
print(df[df["State/UT"] == "Telangana"])

Merged shape: (35, 4)

Telangana row:
     State/UT  Rate of Cognizable Crimes (IPC) (2022)  \
23  Telangana                                   400.1   

    Total Traffic Accidents - Cases  Total Traffic Accidents - Died  
23                            22235                            8184  


## Step 3 — Compute state_risk

**Fix applied**: normalise each column independently FIRST, then combine.
Without this, accident_cases (scale ~10,000s) would dwarf crime_rate (scale ~100-500), 
making the crime weight effectively zero.

**Severity weighting**: fatal accidents weighted 2× case-only accidents.

In [4]:
def minmax_norm(series: pd.Series) -> pd.Series:
    """Normalise to [0, 1]. Returns 0.0 if all values are equal."""
    r = series.max() - series.min()
    if r == 0:
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - series.min()) / r

# Severity-weighted accident signal: fatal accidents count more
df["accident_severity"] = (
    df["Total Traffic Accidents - Cases"] +
    2.0 * df["Total Traffic Accidents - Died"]
)

# Normalise EACH column independently before combining
df["crime_norm"]    = minmax_norm(df["Rate of Cognizable Crimes (IPC) (2022)"])
df["accident_norm"] = minmax_norm(df["accident_severity"])

# Combined risk score (higher = more risky)
df["state_risk"] = 0.6 * df["crime_norm"] + 0.4 * df["accident_norm"]

# Invert to safety direction (1.0 = safest) to match scorer.py convention
df["accident_idx"] = 1.0 - df["state_risk"]

print("Top 10 most dangerous states (lowest accident_idx):")
print(df[["State/UT", "crime_norm", "accident_norm", "state_risk", "accident_idx"]]
      .sort_values("accident_idx")
      .head(10)
      .to_string(index=False))

Top 10 most dangerous states (lowest accident_idx):
      State/UT  crime_norm  accident_norm  state_risk  accident_idx
         Delhi    1.000000       0.100926    0.640370      0.359630
    Tamil Nadu    0.150432       1.000000    0.490259      0.509741
        Kerala    0.446508       0.505519    0.470112      0.529888
Madhya Pradesh    0.219192       0.799058    0.451138      0.548862
 Uttar Pradesh    0.091536       0.934472    0.428710      0.571290
   Maharashtra    0.182853       0.693092    0.386949      0.613051
     Karnataka    0.106405       0.598495    0.303241      0.696759
     Telangana    0.257271       0.365680    0.300635      0.699365
     Rajasthan    0.179952       0.460912    0.292336      0.707664
Andhra Pradesh    0.183941       0.386108    0.264808      0.735192


## Step 4 — Extract Telangana and generate Hyderabad-bounded synthetic points

**Fix applied**:
- Filter to Telangana row only (not all states)
- Constrain lat/lon to Hyderabad city bbox (not ±0.5 deg from state centroid)
- Use Gaussian noise (std=0.05) instead of flat uniform ±0.1 to reduce artificial uniformity

In [5]:
telangana = df[df["State/UT"] == "Telangana"]
assert len(telangana) == 1, "Telangana row not found — check State/UT column values"

base_accident_idx = float(telangana["accident_idx"].iloc[0])
print(f"Telangana base accident_idx: {base_accident_idx:.4f}")
print(f"Generating {N_POINTS} synthetic points within Hyderabad bbox...")

rows = []
for _ in range(N_POINTS):
    lat = np.random.uniform(HYD_BBOX["lat_min"], HYD_BBOX["lat_max"])
    lon = np.random.uniform(HYD_BBOX["lon_min"], HYD_BBOX["lon_max"])

    # Gaussian local variation — smaller std, no clunky hard boundaries
    local_variation = np.random.normal(0, 0.05)
    accident_idx = float(np.clip(base_accident_idx + local_variation, 0.0, 1.0))

    rows.append({
        "lat":          lat,
        "lon":          lon,
        "accident_idx": round(accident_idx, 6),
        "source":       "synthetic_telangana_state",  # provenance column
    })

synthetic_df = pd.DataFrame(rows)
print(f"\nGenerated: {len(synthetic_df)} rows")
print(synthetic_df.describe())

Telangana base accident_idx: 0.6994
Generating 500 synthetic points within Hyderabad bbox...

Generated: 500 rows
              lat         lon  accident_idx
count  500.000000  500.000000    500.000000
mean    17.390469   78.450096      0.703597
std      0.118824    0.142226      0.048241
min     17.202025   78.202879      0.564521
25%     17.286589   78.325003      0.669859
50%     17.384228   78.450101      0.705934
75%     17.487441   78.569069      0.736749
max     17.599765   78.698562      0.827369


## Step 5 — Validate and save

In [6]:
# Sanity checks before saving
assert synthetic_df["accident_idx"].between(0, 1).all(), "accident_idx out of [0,1]"
assert synthetic_df["lat"].between(HYD_BBOX["lat_min"], HYD_BBOX["lat_max"]).all(), "lat out of bbox"
assert synthetic_df["lon"].between(HYD_BBOX["lon_min"], HYD_BBOX["lon_max"]).all(), "lon out of bbox"
assert not synthetic_df.isnull().any().any(), "Null values found"

print("All validations passed.")

OUT_PATH = "hyderabad_accident_idx_synthetic.csv"
synthetic_df.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}")
print(f"\nColumns: {synthetic_df.columns.tolist()}")
print("\nSample rows:")
print(synthetic_df.head())

All validations passed.
Saved to hyderabad_accident_idx_synthetic.csv

Columns: ['lat', 'lon', 'accident_idx', 'source']

Sample rows:
         lat        lon  accident_idx                     source
0  17.349816  78.675357      0.731749  synthetic_telangana_state
1  17.262407  78.277997      0.775517  synthetic_telangana_state
2  17.223233  78.633088      0.778326  synthetic_telangana_state
3  17.208234  78.684955      0.737737  synthetic_telangana_state
4  17.532977  78.306170      0.676194  synthetic_telangana_state


## Usage in scorer.py

```python
# In AccidentFeatureExtractor.extract():
# Load this CSV and do a nearest-neighbour lookup on (lat, lon)
# to get accident_idx for a given segment centroid.
# Replace with PostGIS ST_DWithin query once GHMC data is loaded.

import pandas as pd
from scipy.spatial import KDTree

df = pd.read_csv('hyderabad_accident_idx_synthetic.csv')
tree = KDTree(df[['lat', 'lon']].values)

def lookup_accident_idx(lat, lon):
    _, idx = tree.query([lat, lon], k=3)   # k=3 nearest, average
    return float(df.iloc[idx]['accident_idx'].mean())
```

## TODO — replace synthetic data with real GHMC data

1. Download ward-level accident data from https://data.telangana.gov.in
2. Load into PostGIS: `accidents(id SERIAL, geom GEOMETRY(POINT,4326), occurred_at TIMESTAMP, severity TEXT)`
3. Replace `AccidentFeatureExtractor.extract()` stub with the PostGIS spatial join query
4. Delete this synthetic CSV once PostGIS data covers all Hyderabad segments
